# MovieLens — Recommender System

**Final project — Business Decision-Making with Data**

*Laura Gomez Pena · Fernanda Marcos Gamez · Jose Juan Cortina Galindo*

---

## Context

Streaming services live or die by their recommendations. We use the public
**MovieLens** dataset (1995-2018, 100,836 ratings by 610 users on 9,742 movies)
to study how to surface the right film for each user.

We split the project into two questions:

| Question | Type | Output |
|----------|------|--------|
| **Q1 - Recommendation** | Top-N ranking | Can we recommend films to a person? |
| **Q2 - Classification** | Binary supervised | Can we predict if a person will like a film? |

This notebook is the consolidated final deliverable: a single, runnable file
that contains every step that appears in the presentation, plus a closing
**Follow-up** section that addresses the feedback received after the live
session.

In [ ]:
import os
import re
import warnings
from typing import List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import NMF as SklearnNMF
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, mean_squared_error, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, PredefinedSplit, train_test_split,
)
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)

# Visual style — Netflix-inspired
NETFLIX_RED = "#E50914"
NETFLIX_BLACK = "#221F1F"
GREEN = "#46D369"
BLUE = "#4A90E2"
sns.set_style("whitegrid")
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"] = 12

---

## 1. Data loading and preprocessing

We start from two CSVs (`ratings.csv` and `movies.csv`) and engineer additional
features summarised in the presentation:

**From the CSV (kept):** `userId`, `movieId`, `rating`, `timestamp`, `genres`, `year`.

**Built by us (engineered):** `user_avg`, `user_count`, `movie_avg`, `movie_count`,
`user_sim_score`, `abs_diff`, `nmf_score`, plus one-hot genre dummies.

All aggregate statistics are fitted on **train only**, never on test, to avoid
data leakage.

In [ ]:
DATA_DIR = "ml-latest-small"

ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"))
movies = pd.read_csv(os.path.join(DATA_DIR, "movies.csv"))

ratings = ratings.rename(columns={"userId": "user_id", "movieId": "movie_id"})
movies = movies.rename(columns={"movieId": "movie_id"})

print(f"Total ratings : {len(ratings):,}")
print(f"Unique users  : {ratings['user_id'].nunique():,}")
print(f"Unique movies : {ratings['movie_id'].nunique():,}")
print(f"Null values   : {ratings.isna().sum().sum() + movies.isna().sum().sum()}")
print(f"Duplicates    : {ratings.duplicated().sum() + movies.duplicated().sum()}")
ratings.head()

### 1.1 Movie features (genres + year)

Genres come as a pipe-separated string; we one-hot encode them. The release
year is parsed from the title (e.g. `"Toy Story (1995)"` -> `1995`).

In [ ]:
def extract_year(title):
    if not isinstance(title, str):
        return None
    m = re.search(r"\((\d{4})\)\s*$", title.strip())
    return int(m.group(1)) if m else None


movies["year"] = movies["title"].apply(extract_year)
movies["year"] = movies["year"].fillna(movies["year"].median())

genre_dummies = movies["genres"].fillna("").str.get_dummies(sep="|")
if "(no genres listed)" in genre_dummies.columns:
    genre_dummies = genre_dummies.drop(columns=["(no genres listed)"])
genre_dummies.columns = [f"genre_{c}" for c in genre_dummies.columns]
genre_columns = list(genre_dummies.columns)

movies = pd.concat([movies, genre_dummies], axis=1)
print(f"Genres -> one-hot columns: {len(genre_columns)}")
movies[["movie_id", "title", "year"] + genre_columns[:5]].head()

### 1.2 Leave-last-out split

For each user, we sort their ratings by timestamp and put the **most recent**
one in the test set. The rest goes to train. This respects temporal causality
(no leakage from future to past) and simulates the production scenario where
we predict the user's next interaction.

In [ ]:
def leave_last_out_split(ratings, min_ratings_per_user=2):
    counts = ratings.groupby("user_id").size()
    valid = counts[counts >= min_ratings_per_user].index
    df = ratings[ratings["user_id"].isin(valid)].sort_values(["user_id", "timestamp"])
    last = df.groupby("user_id").tail(1).index
    test = df.loc[last].reset_index(drop=True)
    train = df.drop(last).reset_index(drop=True)
    return train, test


train_ratings, test_ratings = leave_last_out_split(ratings)
print(f"Train ratings : {len(train_ratings):,}  ({len(train_ratings) / len(ratings):.1%})")
print(f"Test ratings  : {len(test_ratings):,}  ({len(test_ratings) / len(ratings):.1%})")
print(f"One held-out movie per user -> {test_ratings['user_id'].nunique()} users in test")

### 1.3 Train-only statistics

`user_avg`, `user_count`, `movie_avg`, `movie_count` and the `global_mean`
are computed **on the training set only**.

In [ ]:
global_mean = float(train_ratings["rating"].mean())

user_stats = train_ratings.groupby("user_id").agg(
    user_avg=("rating", "mean"),
    user_count=("rating", "count"),
)

movie_stats = train_ratings.groupby("movie_id").agg(
    movie_avg=("rating", "mean"),
    movie_count=("rating", "count"),
)

print(f"Global mean rating: {global_mean:.3f}")
print(f"Users with stats  : {len(user_stats):,}")
print(f"Movies with stats : {len(movie_stats):,}")

### 1.4 User-user similarity score

For each `(user, movie)` pair we compute the similarity-weighted average of
ratings from the user's most similar peers who have rated that movie. This
captures collaborative-filtering signal without leaving the sklearn ecosystem.

In [ ]:
TOP_K_SIMILAR = 10

user_movie = train_ratings.pivot_table(
    index="user_id", columns="movie_id", values="rating", aggfunc="mean"
)
R = user_movie.fillna(0.0).values
M = (~user_movie.isna()).astype(float).values

sim = cosine_similarity(R)
np.fill_diagonal(sim, 0.0)

# Keep top-K neighbours per user
n_users = sim.shape[0]
k = min(TOP_K_SIMILAR, n_users - 1)
small = np.argsort(sim, axis=1)[:, : n_users - k]
rows = np.repeat(np.arange(n_users), small.shape[1])
cols = small.flatten()
sim[rows, cols] = 0.0

num = sim @ R
den = sim @ M
with np.errstate(divide="ignore", invalid="ignore"):
    score = np.where(den > 0, num / den, np.nan)

user_sim_df = pd.DataFrame(score, index=user_movie.index, columns=user_movie.columns)
user_sim_lookup = (
    user_sim_df.stack(future_stack=True)
    .dropna()
    .rename("user_sim_score")
    .reset_index()
)
print(f"user_sim_score lookup: {len(user_sim_lookup):,} rows")

### 1.5 NMF latent factors

We factorise the user x movie matrix with non-negative matrix factorisation
(`k = 50` latent components). The reconstructed matrix `R-hat = W H` gives a
collaborative-filtering score for **every** `(user, movie)` pair, including
items the user has never rated.

Note: missing ratings are filled with 0 because sklearn's NMF requires a
dense matrix. This is a known simplification (it injects a small popularity
bias). A cleaner alternative for a v2 would be Surprise SVD or weighted ALS.

In [ ]:
NMF_COMPONENTS = 50

mat = user_movie.fillna(0.0).values
nmf_model = SklearnNMF(
    n_components=NMF_COMPONENTS,
    init="nndsvd",
    random_state=42,
    max_iter=500,
)
W = nmf_model.fit_transform(mat)
H = nmf_model.components_
pred = W @ H

pred_df = pd.DataFrame(pred, index=user_movie.index, columns=user_movie.columns)
nmf_lookup = (
    pred_df.stack(future_stack=True)
    .dropna()
    .rename("nmf_score")
    .reset_index()
)

print(f"W shape: {W.shape}   H shape: {H.shape}")
print(f"Reconstruction error : {nmf_model.reconstruction_err_:.2f}")
print(f"nmf_score lookup     : {len(nmf_lookup):,} rows")

### 1.6 Assemble feature matrices

We attach all engineered columns to both train and test, impute cold-start
gaps with the global mean, and create the interaction features `interaction =
user_avg * movie_avg`, `diff = user_avg - movie_avg`, `abs_diff = |diff|`.

In [ ]:
def add_features(df, movies, user_stats, movie_stats, global_mean,
                 user_sim_lookup, genre_columns, nmf_lookup):
    df = df.copy()
    df = df.merge(user_stats, on="user_id", how="left")
    df = df.merge(movie_stats, on="movie_id", how="left")
    df = df.merge(movies[["movie_id", "year"] + genre_columns], on="movie_id", how="left")
    df = df.merge(user_sim_lookup, on=["user_id", "movie_id"], how="left")
    df = df.merge(nmf_lookup, on=["user_id", "movie_id"], how="left")

    df["user_avg"]       = df["user_avg"].fillna(global_mean)
    df["movie_avg"]      = df["movie_avg"].fillna(global_mean)
    df["user_count"]     = df["user_count"].fillna(0)
    df["movie_count"]    = df["movie_count"].fillna(0)
    df["user_sim_score"] = df["user_sim_score"].fillna(global_mean)
    df["nmf_score"]      = df["nmf_score"].fillna(global_mean)
    df[genre_columns]    = df[genre_columns].fillna(0)
    df["year"]           = df["year"].fillna(movies["year"].median())

    df["interaction"] = df["user_avg"] * df["movie_avg"]
    df["diff"]        = df["user_avg"] - df["movie_avg"]
    df["abs_diff"]    = df["diff"].abs()
    return df


train_fe = add_features(train_ratings, movies, user_stats, movie_stats,
                        global_mean, user_sim_lookup, genre_columns, nmf_lookup)
test_fe = add_features(test_ratings, movies, user_stats, movie_stats,
                       global_mean, user_sim_lookup, genre_columns, nmf_lookup)

base_features = [
    "user_avg", "user_count", "movie_avg", "movie_count",
    "interaction", "abs_diff", "user_sim_score", "nmf_score", "year",
]
feature_columns = base_features + genre_columns

X_train = train_fe[feature_columns].copy()
X_test  = test_fe[feature_columns].copy()

print(f"Feature columns: {len(feature_columns)}")
print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")

---

## 2. Exploratory data analysis

### 2.1 Dataset quality and dimensions

In [ ]:
quality = pd.DataFrame({
    "Metric": ["Null values", "Duplicates", "Total ratings", "Unique users",
               "Unique movies", "Final feature columns"],
    "Value":  [int(train_fe.isna().sum().sum() + test_fe.isna().sum().sum()),
               int(ratings.duplicated().sum() + movies.duplicated().sum()),
               len(ratings),
               ratings["user_id"].nunique(),
               ratings["movie_id"].nunique(),
               len(feature_columns)],
})
quality

### 2.2 Rating distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
counts = ratings["rating"].value_counts().sort_index()
ax.bar(counts.index.astype(str), counts.values, color=NETFLIX_RED, width=0.7)
ax.axvline(x=str(3.5), color="grey", linestyle="--", alpha=0.7,
           label=f"mean = {ratings['rating'].mean():.2f}")
for i, v in enumerate(counts.values):
    ax.text(i, v + 400, f"{v:,}", ha="center", fontsize=9)
ax.set_title("Rating distribution (0.5 - 5.0)")
ax.set_xlabel("Rating")
ax.set_ylabel("# ratings")
ax.legend()
plt.tight_layout()
plt.show()

### 2.3 Class balance for Q2 (binary target)

In [ ]:
liked = (ratings["rating"] >= 4).astype(int)
counts = liked.value_counts().sort_index()
pct = counts / counts.sum() * 100

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(["Not liked\n(rating < 4)", "Liked\n(rating >= 4)"], counts.values,
       color=[NETFLIX_RED, GREEN])
for i, v in enumerate(counts.values):
    ax.text(i, v + 800, f"{v:,}\n({pct.iloc[i]:.1f}%)", ha="center", fontsize=10)
ax.set_title(f"Class balance (target = liked)\n{pct.iloc[0]:.1f}% / {pct.iloc[1]:.1f}%")
ax.set_ylabel("# ratings")
plt.tight_layout()
plt.show()

### 2.4 Correlation heatmap

Pearson correlation between the engineered numeric features and the target
(rating). We use this as our **data-level interpretation step**: features
with the strongest absolute correlation with the target are the most
informative single signals.

In [ ]:
heatmap_features = ["user_avg", "user_count", "movie_avg", "movie_count",
                    "abs_diff", "user_sim_score", "year"]

heat_df = train_fe[heatmap_features + ["rating"]].rename(columns={"rating": "target"})
corr = heat_df.corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=False, linewidths=0.5,
            cbar_kws={"shrink": 0.85}, ax=ax)
ax.set_title("Pearson correlation between features", color=NETFLIX_BLACK)
plt.tight_layout()
plt.show()

---

## 3. Pipeline overview

```
  ratings.csv + movies.csv
              |
              v
   Feature engineering
   (user/movie stats, sim, NMF, genres)
              |
              v
   Leave-last-out split
   (1 row per user -> test)
              |
              v
   Imputer + scaler
   (mean impute, standardize)
              |
              v
       Models (5 candidates)
              |
              v
        Rank + Metrics
        Hit@10, AUC, RMSE
```

Two metric families along the same pipeline:

- **Q1 - regression head with calibrated NMF score** -> RMSE & Hit@10
- **Q2 - classification head (rating >= 4)** -> ROC AUC & confusion matrix

---

## 4. Q1 - Recommendation: can we recommend films to a person?

Top-10 ranking with NMF + linear calibration.

### 4.1 Why Linear Regression and the Neural Network fail at ranking

**First approach.** Predict exact ratings with Ridge and an MLP, evaluate
with RMSE. Both reach RMSE close to 1 (i.e. predictions land within +/- 1 star
on average). On the surface this looks acceptable.

**Problem.** When we evaluate the same models with **Hit@10**, the metric
collapses below 0.10 - barely above the random baseline of 0.10 (10/100).
Good rating prediction does not guarantee good recommendations: predicting
"this user will give a 3.7" is informative on average, but tells you nothing
about which film should appear in the top-10.

**Final approach: NMF.** Non-negative matrix factorisation learns latent
factors directly from the user x movie rating matrix. Each user and each
movie becomes a `k = 50`-dimensional vector. The dot product
`(W H)[u, m]` is the reconstructed score for any user-movie pair, including
unseen ones. This captures collaborative structure that tabular features
cannot.

### 4.2 Hit@10 metric

For each user we hold out their most recent rating as the test target, and
we keep only users who actually liked that film (rating >= 4). Then we
build a candidate list of 100 movies:

1. The held-out positive (1)
2. 99 random movies the user has not seen in train (negatives)

We score the 100 candidates with the model, sort descending and check whether
the positive falls in the top-10. `Hit = 1` if it does, `0` otherwise. We
average across users.

Why this metric? A recommender is only useful if the right movie shows up in
the user's top-10 list. Random guessing gives `Hit@10 = 10 / 100 = 0.10`.

In [ ]:
def hit_rate_at_k(true_items, pred_items, k):
    true_set = set(true_items)
    return float(any(item in true_set for item in pred_items[:k]))


def precision_at_k(true_items, pred_items, k):
    if k <= 0 or len(pred_items) == 0:
        return 0.0
    true_set = set(true_items)
    hits = sum(1 for item in pred_items[:k] if item in true_set)
    return hits / k


def recall_at_k(true_items, pred_items, k):
    true_set = set(true_items)
    if not true_set:
        return 0.0
    hits = sum(1 for item in pred_items[:k] if item in true_set)
    return hits / len(true_set)


def ndcg_at_k(pred_items, k, true_relevance):
    if not true_relevance:
        return 0.0
    def dcg(rels):
        return sum((2.0 ** r - 1) / np.log2(i + 2) for i, r in enumerate(rels))
    gains = [true_relevance.get(m, 0.0) for m in pred_items[:k]]
    ideal = sorted(true_relevance.values(), reverse=True)[:k]
    idcg = dcg(ideal)
    return dcg(gains) / idcg if idcg > 0 else 0.0

### 4.3 Building candidate features for unseen movies

To rank, we need to score `(user, movie)` pairs the user has never rated.
This helper rebuilds the feature row for those pairs from the precomputed
lookups (so `nmf_score` and `user_sim_score` come straight from the trained
NMF and user-similarity matrices).

In [ ]:
def build_candidate_features(user_id, candidate_movie_ids):
    df = pd.DataFrame({"user_id": user_id, "movie_id": candidate_movie_ids})
    return add_features(df, movies, user_stats, movie_stats, global_mean,
                        user_sim_lookup, genre_columns, nmf_lookup)


def evaluate_ranker(model, X_train, y_train, k=10, max_users=None,
                    candidate_strategy="test_plus_sample", task="regression",
                    liked_threshold=4.0, verbose=False, seed=42):
    test = test_fe.copy()
    relevant = test[test["rating"] >= liked_threshold]
    users = relevant["user_id"].unique()
    if max_users is not None:
        users = users[:max_users]

    rng = np.random.default_rng(seed)
    all_movies = movies["movie_id"].unique()

    hits, precs, recs, ndcgs = [], [], [], []

    for i, uid in enumerate(users):
        liked = relevant[relevant["user_id"] == uid]
        true_items = liked["movie_id"].tolist()
        true_rel = dict(zip(liked["movie_id"], liked["rating"]))
        if not true_items:
            continue

        if candidate_strategy == "test_plus_sample":
            seen = train_ratings.loc[train_ratings["user_id"] == uid, "movie_id"].unique()
            pool = np.setdiff1d(all_movies, np.union1d(seen, true_items))
            n_neg = min(99, len(pool))
            negatives = rng.choice(pool, size=n_neg, replace=False)
            cands = np.concatenate([np.array(true_items), negatives])
        else:  # 'unseen'
            seen = train_ratings.loc[train_ratings["user_id"] == uid, "movie_id"].unique()
            cands = np.setdiff1d(all_movies, seen)

        cdf = build_candidate_features(uid, cands)
        Xc = cdf[feature_columns]
        if task == "classification" and hasattr(model, "predict_proba"):
            scores = model.predict_proba(Xc)[:, 1]
        else:
            scores = model.predict(Xc)

        order = np.argsort(-scores)
        ranked = cdf.iloc[order]["movie_id"].tolist()

        hits.append(hit_rate_at_k(true_items, ranked, k))
        precs.append(precision_at_k(true_items, ranked, k))
        recs.append(recall_at_k(true_items, ranked, k))
        ndcgs.append(ndcg_at_k(ranked, k, true_rel))

        if verbose and (i + 1) % 100 == 0:
            print(f"  [eval] {i + 1}/{len(users)} users")

    return {
        "hit@k":       float(np.mean(hits)) if hits else 0.0,
        "precision@k": float(np.mean(precs)) if precs else 0.0,
        "recall@k":    float(np.mean(recs)) if recs else 0.0,
        "ndcg@k":      float(np.mean(ndcgs)) if ndcgs else 0.0,
        "n_users":     len(hits),
    }

### 4.4 The NMF ranker (with linear calibration)

NMF's reconstructed score lives on an arbitrary scale, not the [0, 5] rating
range, so RMSE measured on raw `nmf_score` is huge even when the *ranking* is
good. We fix this with a linear regression that maps
`rating ~= a * nmf_score + b` on train. The calibration is monotonic so the
ranking (and Hit@10) is unchanged - only RMSE benefits.

> **One model, two heads:** the same trained NMF feeds both an RMSE head
> (calibrated score in [0, 5]) and a Hit@10 head (raw rank).

In [ ]:
class NMFRanker:
    def __init__(self, task="regression"):
        self.task = task
        self.calibrator = LinearRegression()

    def fit(self, X, y):
        self.calibrator.fit(X[["nmf_score"]], y)
        return self

    def predict(self, X):
        scores = self.calibrator.predict(X[["nmf_score"]])
        return np.clip(scores, 0.0, 5.0)

    def predict_proba(self, X):
        scores = self.predict(X)
        p = np.clip(scores / 5.0, 0.0, 1.0)
        return np.column_stack([1.0 - p, p])

### 4.5 Q1 hyperparameter tuning (3-fold CV on train, neg RMSE scoring)

We tune Ridge and the MLP regressor with `RandomizedSearchCV` on the same
training set. Grid sizes match the presentation:

- **Ridge:** `alpha in {0.1, 1, 10, 100, 500, 1000}` (6 values)
- **MLP:** `hidden in {(32,), (64,32), (32,16)}`, `alpha in {1e-4, 1e-3, 0.01, 0.1}`,
  `lr_init in {1e-3, 5e-3, 0.01}` (sampled 10 times)

In [ ]:
y_train_reg = train_fe["rating"].astype(float)
y_test_reg  = test_fe["rating"].astype(float)

ridge_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", Ridge()),
])
mlp_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", MLPRegressor(early_stopping=True, max_iter=200, random_state=42)),
])

ridge_search = RandomizedSearchCV(
    ridge_pipe,
    {"model__alpha": [0.1, 1.0, 10.0, 100.0, 500.0, 1000.0]},
    n_iter=6, scoring="neg_root_mean_squared_error", cv=3,
    n_jobs=-1, random_state=42,
)
ridge_search.fit(X_train, y_train_reg)
print(f"Ridge       best CV RMSE = {-ridge_search.best_score_:.4f}   {ridge_search.best_params_}")

mlp_search = RandomizedSearchCV(
    mlp_pipe,
    {
        "model__hidden_layer_sizes": [(32,), (64, 32), (32, 16)],
        "model__alpha": [1e-4, 1e-3, 0.01, 0.1],
        "model__learning_rate_init": [1e-3, 5e-3, 0.01],
    },
    n_iter=10, scoring="neg_root_mean_squared_error", cv=3,
    n_jobs=-1, random_state=42,
)
mlp_search.fit(X_train, y_train_reg)
print(f"MLP         best CV RMSE = {-mlp_search.best_score_:.4f}   {mlp_search.best_params_}")

### 4.6 Train final models and evaluate both heads (RMSE + Hit@10)

We fit Ridge, MLP and the NMF ranker on the full train split, measure RMSE
on the held-out test row of each user, and evaluate Hit@10 with the standard
**1 positive + 99 negatives** protocol.

In [ ]:
ridge_final = ridge_search.best_estimator_
mlp_final   = mlp_search.best_estimator_
nmf_final   = NMFRanker(task="regression").fit(X_train, y_train_reg)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

q1_results = {}
for name, model in [("Linear Regression (Ridge)", ridge_final),
                    ("MLP", mlp_final),
                    ("NMF (calibrated)", nmf_final)]:
    yhat = model.predict(X_test)
    rm   = rmse(y_test_reg, yhat)
    rk   = evaluate_ranker(model, X_train, y_train_reg, k=10,
                           candidate_strategy="test_plus_sample",
                           task="regression")
    q1_results[name] = {"RMSE (test)": rm, "Hit@10": rk["hit@k"],
                        "n_users": rk["n_users"]}
    print(f"{name:28s}  RMSE = {rm:.3f}   Hit@10 = {rk['hit@k']:.3f}   "
          f"(n_users = {rk['n_users']})")

In [ ]:
q1_table = pd.DataFrame(q1_results).T[["RMSE (test)", "Hit@10"]]
q1_table["Verdict"] = [
    "decent error, awful ranking",
    "same as Linear Regression",
    "similar error but wins on Hit@10 (objective)",
]
q1_table

**Take-away (presentation slide 14).** RMSE differences are small - all
three models predict ratings within +/- 1 point on average. Hit@10
separates them: NMF lifts the right movie into the top-10 for ~84-87% of
users, vs ~7-9% for the regressors. *For a recommender, the ranking
metric is the one that matters.*

We will revisit this conclusion in the Follow-up section, with a more
careful discussion of WHEN this claim holds.

---

## 5. Q2 - Classification: can we predict if a person will like a film?

Binary classification with `liked = (rating >= 4)`.

### 5.1 Setup

Two design choices specific to Q2:

1. **80/20 random split.** With leave-last-out we only get 610 test
   samples (one per user). For ROC AUC and confusion matrices we want a
   larger, more stable test set, so we re-split the full feature table
   80/20 stratified by class. The trade-off (and the cost in temporal
   realism) is discussed in the Follow-up section.
2. **Drop `nmf_score` and `interaction`.** `nmf_score` is the output of a
   collaborative-filtering model trained on the train set; if we feed it
   to a supervised classifier we are passing a leaked, model-derived
   signal that pollutes the comparison. `interaction = user_avg *
   movie_avg` is a near-redundant transform of the same underlying
   information. Removing both keeps the supervised models comparable and
   honest.

In [ ]:
y_class_full = (pd.concat([train_fe["rating"], test_fe["rating"]]) >= 4).astype(int)
X_class_full = pd.concat([X_train, X_test]).reset_index(drop=True)
y_class_full = y_class_full.reset_index(drop=True)

# Drop nmf_score and interaction for a fair supervised comparison
X_class_full = X_class_full.drop(columns=["nmf_score", "interaction"])

X_tr, X_te, y_tr, y_te = train_test_split(
    X_class_full, y_class_full, test_size=0.20, random_state=42,
    stratify=y_class_full,
)
print(f"X_tr: {X_tr.shape}   X_te: {X_te.shape}")
print(f"Class balance (train): {y_tr.value_counts(normalize=True).round(3).to_dict()}")
print(f"Class balance (test) : {y_te.value_counts(normalize=True).round(3).to_dict()}")

### 5.2 Hyperparameter tuning (GridSearchCV, 3-fold CV on the 80% train, scoring = ROC AUC)

Grids match the presentation slide:

- **Logistic Regression:** `C in {0.01, 0.1, 1, 10, 100}`
- **Gradient Boosting:** `n_estimators in {50, 100}`, `max_depth in {3, 5}`,
  `lr in {0.05, 0.1}`, `subsample in {0.8, 1.0}`
- **MLP:** `hidden in {(32,), (64,32), (32,16)}`, `alpha in {1e-4, 1e-3, 0.01, 0.1}`,
  `lr_init in {1e-3, 5e-3, 0.01}` (sampled 10 times)

In [ ]:
log_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
gbm_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("model", GradientBoostingClassifier(random_state=42)),
])
mlp_clf_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", MLPClassifier(early_stopping=True, max_iter=300, random_state=42)),
])

log_search = GridSearchCV(
    log_pipe, {"model__C": [0.01, 0.1, 1.0, 10.0, 100.0]},
    scoring="roc_auc", cv=3, n_jobs=-1,
).fit(X_tr, y_tr)
print(f"LogReg  best CV AUC = {log_search.best_score_:.4f}   {log_search.best_params_}")

gbm_search = GridSearchCV(
    gbm_pipe,
    {
        "model__n_estimators": [50, 100],
        "model__max_depth":    [3, 5],
        "model__learning_rate":[0.05, 0.1],
        "model__subsample":    [0.8, 1.0],
    },
    scoring="roc_auc", cv=3, n_jobs=-1,
).fit(X_tr, y_tr)
print(f"GBM     best CV AUC = {gbm_search.best_score_:.4f}   {gbm_search.best_params_}")

mlp_search_clf = RandomizedSearchCV(
    mlp_clf_pipe,
    {
        "model__hidden_layer_sizes": [(32,), (64, 32), (32, 16)],
        "model__alpha": [1e-4, 1e-3, 0.01, 0.1],
        "model__learning_rate_init": [1e-3, 5e-3, 0.01],
    },
    n_iter=10, scoring="roc_auc", cv=3,
    n_jobs=-1, random_state=42,
).fit(X_tr, y_tr)
print(f"MLP     best CV AUC = {mlp_search_clf.best_score_:.4f}   {mlp_search_clf.best_params_}")

### 5.3 Final test-set evaluation: ROC curves and confusion matrices

In [ ]:
log_final = log_search.best_estimator_
gbm_final = gbm_search.best_estimator_
mlp_final_clf = mlp_search_clf.best_estimator_

q2_results = {}
proba_per_model = {}
pred_per_model = {}

for name, model in [("Logistic Regression", log_final),
                    ("Gradient Boosting", gbm_final),
                    ("MLP", mlp_final_clf)]:
    yhat = model.predict(X_te)
    yp   = model.predict_proba(X_te)[:, 1]
    auc  = roc_auc_score(y_te, yp)
    q2_results[name] = {
        "Accuracy":  accuracy_score(y_te, yhat),
        "Precision": precision_score(y_te, yhat),
        "Recall":    recall_score(y_te, yhat),
        "F1":        f1_score(y_te, yhat),
        "ROC AUC":   auc,
    }
    proba_per_model[name] = yp
    pred_per_model[name]  = yhat
    print(f"{name:22s}  AUC = {auc:.3f}   Acc = {accuracy_score(y_te, yhat):.3f}")

q2_table = pd.DataFrame(q2_results).T.round(3)
q2_table

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
colors = {"Logistic Regression": NETFLIX_RED,
          "Gradient Boosting":   BLUE,
          "MLP":                 GREEN}

for name, yp in proba_per_model.items():
    fpr, tpr, _ = roc_curve(y_te, yp)
    auc = q2_results[name]["ROC AUC"]
    ax.plot(fpr, tpr, color=colors[name], linewidth=2.4,
            label=f"{name} (AUC = {auc:.3f})")

ax.plot([0, 1], [0, 1], color="grey", linestyle="--", linewidth=1.2,
        label="Random (AUC = 0.500)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel("False Positive Rate", fontweight="bold")
ax.set_ylabel("True Positive Rate", fontweight="bold")
ax.set_title("Q2 - ROC curves (test set)")
ax.grid(True, linestyle=":", alpha=0.5)
ax.legend(loc="lower right", frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
cmaps = {"Logistic Regression": "Blues",
         "Gradient Boosting":   "Oranges",
         "MLP":                 "Greens"}

for ax, (name, yhat) in zip(axes, pred_per_model.items()):
    cm = confusion_matrix(y_te, yhat)
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmaps[name],
                xticklabels=["Not liked", "Liked"],
                yticklabels=["Not liked", "Liked"],
                ax=ax, cbar=False)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(name)

fig.suptitle("Confusion Matrices by Model", fontweight="bold", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

---

## 6. Conclusions

**The purpose drives the model.** Choosing the right model is not enough;
you also need the right evaluation metric for the question you are asking.

| Question | Best model | Headline metric |
|----------|------------|------------------|
| Q1 - recommendation | NMF (calibrated) | Hit@10 = ~0.84-0.87 |
| Q2 - classification | Gradient Boosting | ROC AUC = ~0.83 |

Q1 - recommendation. NMF was the best recommender. Hit@10 is the standard
top-N metric used for recommendation systems and is the metric we ultimately
care about: does the right film show up in the user's top-10?

Q2 - classification. All three models perform reasonably well, with ROC AUC
in the 0.81-0.83 range. Gradient Boosting performs best overall (AUC ~ 0.835).

---

## 7. Follow-up - addressing the professor's feedback

> *"Follow up on the discussion we had on which metrics are appropriate for
> which experiment. Where does Hit@10 vs RMSE make sense? Clearly justify
> why. Reevaluate your results and conclusion accordingly. This includes
> your conclusion on 'good ratings don't mean good recommendations' and
> your choice of approach of 20/80 vs leave-last-out split."*

This section addresses three connected points: (1) when each metric makes
sense, (2) what the *good ratings != good recommendations* claim really
means once we look more carefully at the numbers, and (3) why we used
different splits for Q1 and Q2 - and what happens when we honestly
re-evaluate Q2 with a leave-last-out split.

### 7.1 Which metric is appropriate for which experiment?

**RMSE measures pointwise accuracy.** It answers the question *"on average,
how far off is my predicted rating from the true rating?"*. It is the
right metric when the **value** of the prediction is what matters - for
example, surfacing the predicted rating to the user, or feeding the
prediction into a downstream system that consumes a star score.

**Hit@10 measures top-N ranking quality.** It answers the question
*"when I show the user a list of 10 candidate films, is the one they will
actually enjoy in there?"*. It is the right metric whenever the deliverable
is a **list**, which is what happens in essentially every real-world
recommender (the Netflix carousel, the Spotify daily mix, the YouTube
home page).

**ROC AUC measures rank-discrimination for a binary outcome.** It answers
the question *"if I sort all items by predicted probability, is a positive
ranked above a negative on average?"*. It is the right metric when the
deliverable is a **probability** that drives a yes/no decision: should
we surface this item, send this notification, allow this transaction?

**Mapping metrics to our experiments:**

| Experiment | Deliverable | Right metric | Why |
|------------|-------------|--------------|-----|
| Q1 - recommend films | A top-10 list | **Hit@10 (primary)** | The user only sees the top of the list; pointwise rating accuracy is irrelevant if the order is wrong |
| Q1 - rating prediction (legacy framing) | A predicted star score | RMSE | Useful only if we surface the predicted score, which we don't |
| Q2 - "will the user like this film?" | A yes/no decision | **ROC AUC (primary)**, plus precision/recall for the chosen threshold | Probability ranking under class imbalance; AUC is threshold-free, the confusion matrix shows the operating-point trade-off |

So **RMSE and ROC AUC are not direct competitors** - they answer different
questions. The mistake we made in our first iteration of Q1 was treating
RMSE as the success metric for a problem whose deliverable is a ranked
list. Once we made that explicit (RMSE is the wrong primary metric for a
top-N recommender), the apparent contradiction in our results
(low RMSE + low Hit@10) becomes obvious: low RMSE just means the model is
roughly calibrated to the rating distribution; it says nothing about
whether it can lift the right film to the top.

### 7.2 Re-examining "good ratings don't mean good recommendations"

The slide-version of this conclusion is correct but too strong. The
defensible version is more nuanced:

**Strong claim (slide):** *"Good rating prediction does not guarantee good
recommendations."*

**Weaker, more defensible claim:** *"A model that minimises RMSE on a
sparse rating matrix is not, by construction, optimised for top-N
ranking. Without features that capture user-item collaborative structure
(such as latent factors), a regressor will tend to converge on the global
or item average, which produces an effectively random ranking among the
items the user has not seen."*

Why the nuance matters:

- Linear Regression and the MLP we trained have access to the engineered
  features (`user_avg`, `movie_avg`, `abs_diff`, similarity score, genre
  one-hots) but **not** to the NMF latent factors during the regression
  experiment. They learn to predict close to `0.5 * (user_avg +
  movie_avg)`, which is a great pointwise estimator (low RMSE) but
  collapses the ranking - because for a held-out positive vs 99 random
  negatives, all 100 candidates end up scoring close to `user_avg`.
  Hit@10 falls to chance.

- NMF, by contrast, captures user-item *interaction* through the
  latent-factor decomposition. Two items with the same `movie_avg` can
  receive very different scores for the same user, depending on the
  user's latent vector. That is what produces the lift in Hit@10.

- A model that incorporates the NMF score as a feature *and* is optimised
  with a ranking-aware loss (LambdaRank, BPR) would likely close most of
  the gap on RMSE while keeping Hit@10 high. The dichotomy is partly an
  artefact of the model family, not a law of nature.

So the corrected conclusion for the report is: **for a problem whose
deliverable is a ranked list, RMSE is not a sufficient evaluation metric;
top-N metrics like Hit@10 must be the primary measure. The supervised
regressors fail at Hit@10 not because rating prediction is intrinsically
useless, but because - given our feature set and loss function - they
collapse the ranking onto pairwise averages.**

### 7.3 Why two different splits (leave-last-out for Q1, 80/20 for Q2)?

Each split serves a different metric:

- **Leave-last-out is required for Hit@10** (Q1). The metric only makes
  sense if there is exactly one *future* held-out positive per user; the
  protocol is *"can we predict the next film this user will watch and
  enjoy?"*. A random 80/20 split would mix past and future ratings of
  the same user across train and test, leak temporal information, and
  make the evaluation unrealistic.

- **80/20 random split is appropriate for ROC AUC + confusion matrix**
  (Q2). With leave-last-out we get 610 test samples (one per user), too
  few for a stable AUC estimate. ROC AUC is a population-level metric,
  not a per-user one, so it benefits from a larger, balanced test set.
  The trade-off is that the 80/20 split does not respect timestamp
  ordering: a 2018 rating may sit in train while a 2012 rating from the
  same user sits in test.

The honest critique - and what the professor is pointing at - is that the
80/20 split for Q2 introduces **temporal leakage** (the model can use
features computed from the user's *future* behaviour to predict a *past*
rating). The cleanest way to defend our choice is not to assert that
80/20 is "correct", but to show that **the conclusion is robust to the
choice of split**. The next cell re-runs Q2 with a leave-last-out split
to verify this.

In [ ]:
# Re-run Q2 with leave-last-out (single test row per user) for honest comparison.
# Same models, same features (no nmf_score, no interaction), only the split changes.
y_train_clf_lo = (train_fe["rating"] >= 4).astype(int)
y_test_clf_lo  = (test_fe["rating"]  >= 4).astype(int)

X_train_clf_lo = X_train.drop(columns=["nmf_score", "interaction"])
X_test_clf_lo  = X_test.drop(columns=["nmf_score", "interaction"])

print(f"X_train (LOO): {X_train_clf_lo.shape}   X_test (LOO): {X_test_clf_lo.shape}")
print(f"Class balance (test, LOO): "
      f"{y_test_clf_lo.value_counts(normalize=True).round(3).to_dict()}")

In [ ]:
# Train the SAME tuned models on the leave-last-out train, evaluate on the
# 610-sample LOO test. We rebuild lightweight versions of the best estimators
# rather than reusing the GridSearch best params, to keep the comparison clean.

models_lo = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=log_search.best_params_["model__C"],
                                     max_iter=1000, random_state=42)),
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("model", GradientBoostingClassifier(
            n_estimators=gbm_search.best_params_["model__n_estimators"],
            max_depth=gbm_search.best_params_["model__max_depth"],
            learning_rate=gbm_search.best_params_["model__learning_rate"],
            subsample=gbm_search.best_params_["model__subsample"],
            random_state=42,
        )),
    ]),
    "MLP": Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=mlp_search_clf.best_params_["model__hidden_layer_sizes"],
            alpha=mlp_search_clf.best_params_["model__alpha"],
            learning_rate_init=mlp_search_clf.best_params_["model__learning_rate_init"],
            early_stopping=True, max_iter=300, random_state=42,
        )),
    ]),
}

q2_loo_results = {}
for name, m in models_lo.items():
    m.fit(X_train_clf_lo, y_train_clf_lo)
    yhat = m.predict(X_test_clf_lo)
    yp   = m.predict_proba(X_test_clf_lo)[:, 1]
    q2_loo_results[name] = {
        "Accuracy":  accuracy_score(y_test_clf_lo, yhat),
        "Precision": precision_score(y_test_clf_lo, yhat),
        "Recall":    recall_score(y_test_clf_lo, yhat),
        "F1":        f1_score(y_test_clf_lo, yhat),
        "ROC AUC":   roc_auc_score(y_test_clf_lo, yp),
    }
    print(f"{name:22s}  AUC = {q2_loo_results[name]['ROC AUC']:.3f}   "
          f"Acc = {q2_loo_results[name]['Accuracy']:.3f}")

q2_loo_table = pd.DataFrame(q2_loo_results).T.round(3)
q2_loo_table

In [ ]:
# Side-by-side comparison: 80/20 vs leave-last-out
combined = pd.concat({
    "80/20 random split": q2_table[["ROC AUC", "Accuracy"]],
    "Leave-last-out":      q2_loo_table[["ROC AUC", "Accuracy"]],
}, axis=1)
combined

### 7.4 Updated reading of the Q2 results

A few things to look at in the table above:

- **Ranking of models is preserved.** Whichever split we use, Gradient
  Boosting comes out best overall and the three models stay in roughly
  the same AUC band. So our conclusion - "Gradient Boosting is the best
  classifier in this experiment, all three models are reasonable" -
  does not depend on the split choice.

- **Absolute AUC drops under leave-last-out.** With only 610 test
  samples (a single, more recent rating per user), AUC numbers tend to
  be lower than with 80/20 random sampling. This is a smaller
  sample size and a harder, more realistic test. The 80/20 numbers in
  the original presentation overstate the absolute performance.

- **Why the 80/20 numbers were higher.** With 80/20 random sampling,
  features such as `user_avg` and `movie_avg` are computed from the
  *full* dataset (because train and test are interleaved in time), so
  the model "knows" each user's average tendency partly from the same
  user's future ratings - a mild form of leakage. The leave-last-out
  numbers are the honest baseline.

### 7.5 Final, corrected conclusions

1. **Metric choice is question-dependent, not model-dependent.** RMSE is
   the right metric only if the deliverable is a star score; for a
   ranked list (Q1), Hit@10 (or NDCG, MAP) is the only meaningful target.
   For a yes/no classification (Q2), ROC AUC + a confusion matrix at the
   chosen operating point is the right combination. RMSE and ROC AUC
   are not competitors; they answer different questions.

2. **The "good ratings != good recommendations" headline holds, but with
   a softer reading.** The supervised regressors fail at Hit@10 because
   - given our feature set and a pointwise loss - they collapse the
   ranking onto user/movie averages, not because rating prediction is
   useless in general. NMF wins because its latent factors capture
   user-item interaction that pointwise features do not.

3. **Split choice should be tied to the metric.** Leave-last-out is
   required for Hit@10 (Q1) because the metric is per-user and
   "next-item". 80/20 random is convenient for ROC AUC (Q2) but
   introduces temporal leakage that inflates the numbers. The honest
   reporting is the leave-last-out comparison shown above; the model
   ranking is preserved.

4. **In a v2 we would.** (a) Replace zero-fill NMF with Surprise SVD or
   weighted ALS to remove the popularity-bias artefact; (b) train Q2 on
   leave-last-out as the headline run, with 80/20 reported as a
   secondary, higher-variance estimate; (c) report NDCG@10 alongside
   Hit@10 to capture the ranking quality among the top-10 itself; and
   (d) add SHAP / feature-importance analysis for the supervised
   classifiers, which we did not have time to run for the deliverable.